UPODATED DM-TEST:

In [34]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from statsmodels.stats.sandwich_covariance import cov_hac

def load_and_clean_residual_file(file_path, model_name, file_type="csv"):
    if file_type == "excel":
        df = pd.read_excel(file_path)
    else:
        df = pd.read_csv(file_path)

    df = df.copy()

    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    rename_map = {}
    if "actual_ret_1w" in df.columns:
        rename_map["actual_ret_1w"] = "actual"
    if "predicted_ret_1w" in df.columns:
        rename_map["predicted_ret_1w"] = "prediction"
    if "pred" in df.columns:
        rename_map["pred"] = "prediction"

    df = df.rename(columns=rename_map)

    df["week"] = pd.to_datetime(
        df["week"],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

    needed_cols = ["coin", "week", "residual"]
    missing_cols = [col for col in needed_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{model_name} is missing required columns: {missing_cols}")

    df = df[needed_cols].copy()
    df["model"] = model_name
    df = df.sort_values(["coin", "week"]).reset_index(drop=True)

    dup_count = df.duplicated(["coin", "week"]).sum()
    if dup_count > 0:
        raise ValueError(f"{model_name} has duplicate coin-week rows")

    return df

In [35]:
ols_df = load_and_clean_residual_file("outputs/residuals/ols_residuals.csv", "ols", file_type="csv")
arima_df = load_and_clean_residual_file("outputs/residuals/arima_residuals.csv", "arima", file_type="csv")
fixed_effects_df = load_and_clean_residual_file("outputs/residuals/fe_residuals.csv", "fixed_effects", file_type="csv")
pls_df = load_and_clean_residual_file("outputs/residuals/pls_residuals.csv", "pls", file_type="excel")
rf_df = load_and_clean_residual_file("outputs/residuals/rf_residuals.csv", "rf", file_type="csv")

In [36]:
def dm_test(e1, e2):
    e1 = np.asarray(e1, dtype=float)
    e2 = np.asarray(e2, dtype=float)

    keep = np.isfinite(e1) & np.isfinite(e2)
    e1 = e1[keep]
    e2 = e2[keep]

    d = e1**2 - e2**2
    t = len(d)

    if t < 5:
        return {
            "dm_stat": np.nan,
            "p_value": np.nan,
            "significant": False
        }

    if np.var(d, ddof=1) == 0:
        return {
            "dm_stat": np.nan,
            "p_value": np.nan,
            "significant": False
        }

    q = int(np.floor(t ** (1/3)))

    x = np.ones((t, 1))
    fit = OLS(d, x).fit()

    v_hac = cov_hac(fit, nlags=q)
    se_hac = np.sqrt(v_hac[0, 0])

    dm_stat = fit.params[0] / se_hac

    # one-sided alternative: challenger (model 2) better than benchmark (model 1)
    p_value = 1 - norm.cdf(dm_stat)

    return {
        "dm_stat": dm_stat,
        "p_value": p_value,
        "significant": bool(p_value < 0.05)
    }

In [37]:
def compare_vs_benchmark_by_coin(benchmark_df, challenger_df, benchmark_name="ols", challenger_name="challenger"):
    merged = benchmark_df[["coin", "week", "residual"]].merge(
        challenger_df[["coin", "week", "residual"]],
        on=["coin", "week"],
        how="inner",
        suffixes=("_bench", "_challenger")
    )

    rows = []

    for coin, g in merged.groupby("coin"):
        ans = dm_test(g["residual_bench"], g["residual_challenger"])

        rows.append({
            "coin": coin,
            "Model 1": benchmark_name,
            "Model 2": challenger_name,
            "DM statistic": np.round(ans["dm_stat"], 3) if pd.notna(ans["dm_stat"]) else np.nan,
            "p-value": np.round(ans["p_value"], 3) if pd.notna(ans["p_value"]) else np.nan,
            "Significant": ans["significant"]
        })

    return pd.DataFrame(rows)

Run all 4 model comparisons against OLS benchmark 

In [38]:
dm_arima = compare_vs_benchmark_by_coin(
    benchmark_df=ols_df,
    challenger_df=arima_df,
    benchmark_name="ols",
    challenger_name="arima"
)

dm_fixed_effects = compare_vs_benchmark_by_coin(
    benchmark_df=ols_df,
    challenger_df=fixed_effects_df,
    benchmark_name="ols",
    challenger_name="fixed_effects"
)

dm_pls = compare_vs_benchmark_by_coin(
    benchmark_df=ols_df,
    challenger_df=pls_df,
    benchmark_name="ols",
    challenger_name="pls"
)

dm_rf = compare_vs_benchmark_by_coin(
    benchmark_df=ols_df,
    challenger_df=rf_df,
    benchmark_name="ols",
    challenger_name="rf"
)

dm_results_df = pd.concat(
    [dm_arima, dm_fixed_effects, dm_pls, dm_rf],
    ignore_index=True
).sort_values(["coin", "Model 2"]).reset_index(drop=True)

dm_results_df.to_excel("outputs/dm_test_results_by_coin.xlsx", index=False)
dm_results_df

,coin,Model 1,Model 2,DM statistic,p-value,Significant
0,BCH,ols,arima,0.683,0.247,False
1,BCH,ols,fixed_effects,0.852,0.197,False
2,BCH,ols,pls,2.892,0.002,True
3,BCH,ols,rf,2.983,0.001,True
4,BTC,ols,arima,1.205,0.114,False
5,BTC,ols,fixed_effects,0.415,0.339,False
6,BTC,ols,pls,2.021,0.022,True
7,BTC,ols,rf,1.986,0.023,True
8,DASH,ols,arima,0.623,0.267,False
9,DASH,ols,fixed_effects,2.030,0.021,True


Compare 4 models against each other (excluding OLS benchmark)

In [39]:
from itertools import combinations

def compare_model_pair_by_coin(model_1_df, model_2_df, model_1_name, model_2_name):
    merged = model_1_df[["coin", "week", "residual"]].merge(
        model_2_df[["coin", "week", "residual"]],
        on=["coin", "week"],
        how="inner",
        suffixes=("_1", "_2")
    )

    rows = []

    for coin, g in merged.groupby("coin"):
        ans = dm_test(g["residual_1"], g["residual_2"])

        rows.append({
            "coin": coin,
            "Model 1": model_1_name,
            "Model 2": model_2_name,
            "DM statistic": np.round(ans["dm_stat"], 3) if pd.notna(ans["dm_stat"]) else np.nan,
            "p-value": np.round(ans["p_value"], 3) if pd.notna(ans["p_value"]) else np.nan,
            "Significant": ans["significant"]
        })

    return pd.DataFrame(rows)

In [40]:
non_benchmark_models = {
    "arima": arima_df,
    "fixed_effects": fixed_effects_df,
    "pls": pls_df,
    "rf": rf_df
}

pairwise_results = []

for model_1_name, model_2_name in combinations(non_benchmark_models.keys(), 2):
    res = compare_model_pair_by_coin(
        model_1_df=non_benchmark_models[model_1_name],
        model_2_df=non_benchmark_models[model_2_name],
        model_1_name=model_1_name,
        model_2_name=model_2_name
    )
    pairwise_results.append(res)

dm_non_benchmark_df = pd.concat(pairwise_results, ignore_index=True)
dm_non_benchmark_df = dm_non_benchmark_df.sort_values(["coin", "Model 1", "Model 2"]).reset_index(drop=True)

dm_non_benchmark_df

,coin,Model 1,Model 2,DM statistic,p-value,Significant
0,BCH,arima,fixed_effects,0.507,0.306,False
1,BCH,arima,pls,3.241,0.001,True
2,BCH,arima,rf,3.266,0.001,True
3,BCH,fixed_effects,pls,3.682,0.000,True
4,BCH,fixed_effects,rf,3.792,0.000,True
...,...,...,...,...,...,...
67,ZEC,arima,pls,-1.592,0.944,False
68,ZEC,arima,rf,-0.465,0.679,False
69,ZEC,fixed_effects,pls,-1.594,0.945,False
70,ZEC,fixed_effects,rf,-0.015,0.506,False


In [44]:
def add_better_model_column(df):
    df = df.copy()

    def better_model_row(row):
        if pd.isna(row["p-value"]) or pd.isna(row["DM statistic"]):
            return np.nan
        if row["Significant"] is not True:
            return np.nan
        return row["Model 2"]

    df["better_model"] = df.apply(better_model_row, axis=1)
    return df

dm_non_benchmark_df = add_better_model_column(dm_non_benchmark_df)
dm_non_benchmark_df

,coin,Model 1,Model 2,DM statistic,p-value,Significant,better_model
0,BCH,arima,fixed_effects,0.507,0.306,False,NaN
1,BCH,arima,pls,3.241,0.001,True,pls
2,BCH,arima,rf,3.266,0.001,True,rf
3,BCH,fixed_effects,pls,3.682,0.000,True,pls
4,BCH,fixed_effects,rf,3.792,0.000,True,rf
...,...,...,...,...,...,...,...
67,ZEC,arima,pls,-1.592,0.944,False,NaN
68,ZEC,arima,rf,-0.465,0.679,False,NaN
69,ZEC,fixed_effects,pls,-1.594,0.945,False,NaN
70,ZEC,fixed_effects,rf,-0.015,0.506,False,NaN


In [45]:
dm_non_benchmark_df.to_excel("outputs/dm_test_non_benchmark_models.xlsx", index=False)
print("saved: ../outputs/dm_test_non_benchmark_models.xlsx")

saved: ../outputs/dm_test_non_benchmark_models.xlsx
